In [1]:
# Task 3: Real-Time Echocardiogram Video Analysis
# Put a short ultrasound video at: data/echo.mp4
# Press Q to close the video window.

import os
import cv2
import numpy as np

VIDEO_PATH = "data/echo.mp4"
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
def color_balance(image):
    """Simple gray-world color balance for a BGR frame."""
    result = image.astype(np.float32)
    channel_means = result.mean(axis=(0, 1))
    target_mean = channel_means.mean()
    scale = target_mean / (channel_means + 1e-6)
    return np.clip(result * scale, 0, 255).astype(np.uint8)


def log_transform(image):
    image_float = image.astype(np.float32)
    max_value = np.max(image_float)

    if max_value == 0:
        return np.zeros_like(image, dtype=np.uint8)

    c = 255 / np.log1p(max_value)
    return np.uint8(
        np.clip(c * np.log1p(image_float), 0, 255)
    )


def power_law_transform(image, gamma=1.3):
    image_float = image.astype(np.float32) / 255.0
    return np.uint8(
        255 * np.power(image_float, gamma)
    )


def resize_keep_aspect(image, target_width, target_height):
    """
    Resize image to fit completely inside target dimensions
    without cropping or distortion.
    """
    h, w = image.shape[:2]

    scale = min(target_width / w, target_height / h)

    new_w = max(1, int(w * scale))
    new_h = max(1, int(h * scale))

    resized = cv2.resize(
        image,
        (new_w, new_h),
        interpolation=cv2.INTER_AREA
    )

    # Create black background
    if image.ndim == 3:
        canvas = np.zeros(
            (target_height, target_width, 3),
            dtype=np.uint8
        )
    else:
        canvas = np.zeros(
            (target_height, target_width),
            dtype=np.uint8
        )

    # Center the resized image
    x_offset = (target_width - new_w) // 2
    y_offset = (target_height - new_h) // 2

    canvas[
        y_offset:y_offset + new_h,
        x_offset:x_offset + new_w
    ] = resized

    return canvas



In [3]:

# --------------------------------------------------
# Load video
# --------------------------------------------------

capture = cv2.VideoCapture(VIDEO_PATH)

if not capture.isOpened():
    raise FileNotFoundError(
        f"Video could not be opened: {VIDEO_PATH}"
    )

saved_preview = False

# Display size for each panel
PANEL_WIDTH = 640
PANEL_HEIGHT = 480


# Create resizable OpenCV window
cv2.namedWindow(
    "Raw and Enhanced Echocardiogram",
    cv2.WINDOW_NORMAL
)

cv2.resizeWindow(
    "Raw and Enhanced Echocardiogram",
    PANEL_WIDTH * 2,
    PANEL_HEIGHT
)



In [4]:
# --------------------------------------------------
# Real-time processing
# --------------------------------------------------

while True:

    success, frame = capture.read()

    if not success:
        break

    # 1. Convert frame to grayscale
    gray = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2GRAY
    )

    # 2. Histogram equalization
    equalized = cv2.equalizeHist(gray)

    # 3. JET color mapping
    heatmap = cv2.applyColorMap(
        equalized,
        cv2.COLORMAP_JET
    )

    # 4. Mathematical color balance
    balanced = color_balance(heatmap)

    # 5. Logarithmic transformation
    log_frame = log_transform(balanced)

    # 6. Power-law transformation
    enhanced = power_law_transform(
        log_frame,
        gamma=1.3
    )

    # --------------------------------------------------
    # Resize both images without cropping
    # --------------------------------------------------

    raw_display = resize_keep_aspect(
        frame,
        PANEL_WIDTH,
        PANEL_HEIGHT
    )

    enhanced_display = resize_keep_aspect(
        enhanced,
        PANEL_WIDTH,
        PANEL_HEIGHT
    )

    # --------------------------------------------------
    # Add labels
    # --------------------------------------------------

    cv2.putText(
        raw_display,
        "RAW",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (0, 255, 0),
        2,
        cv2.LINE_AA
    )

    cv2.putText(
        enhanced_display,
        "ENHANCED",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 255, 255),
        2,
        cv2.LINE_AA
    )

    # --------------------------------------------------
    # Side-by-side comparison
    # --------------------------------------------------

    comparison = cv2.hconcat([
        raw_display,
        enhanced_display
    ])

    # --------------------------------------------------
    # Save first comparison frame
    # --------------------------------------------------

    if not saved_preview:

        output_path = os.path.join(
            OUTPUT_DIR,
            "echo_side_by_side.png"
        )

        cv2.imwrite(
            output_path,
            comparison
        )

        print(
            f"First comparison frame saved to: {output_path}"
        )

        saved_preview = True

    # --------------------------------------------------
    # Display
    # --------------------------------------------------

    cv2.imshow(
        "Raw and Enhanced Echocardiogram",
        comparison
    )

    # Press Q to quit
    if cv2.waitKey(25) & 0xFF == ord("q"):
        break

First comparison frame saved to: output\echo_side_by_side.png


In [5]:
# --------------------------------------------------
# Cleanup
# --------------------------------------------------

capture.release()
cv2.destroyAllWindows()

print("Video processing completed.")

Video processing completed.
